# 🚀 Train Your Own Antigravity Coding Agent (Continual Learning SFT)

<a href="https://colab.research.google.com/github/tolani007/sft-coding-agent/blob/main/notebooks/sft_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook trains a coding agent on exactly **6,625 real developer traces** (including your own Antigravity sessions). We'll use **Google Gemma-4-4b-it** and **Unsloth** for blazing fast 2x training on a free T4 GPU.

### Pre-requisites:
1. Go to **Runtime > Change runtime type** and select **T4 GPU**.
2. Add your Hugging Face Token to Colab Secrets: Click the 🔑 (Secrets) icon on the left sidebar, add a secret named `HF_TOKEN`, and toggle "Notebook access" to ON.

## 1. Install Dependencies

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install datasets

## 2. Authenticate with Hugging Face

In [ ]:
from google.colab import userdata
import os

# Load token from Colab Secrets
hf_token = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = hf_token

from huggingface_hub import login
login(token=hf_token)

## 3. Load the Model via Unsloth
We load `google/gemma-4-4b-it` in 4-bit precision to fit it on the free 16GB GPU.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 4096 # Fits in T4 memory
dtype = None # Auto detects fp16/bf16
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "google/gemma-4-4b-it",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Attach LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0, 
    bias = "none",    
    use_gradient_checkpointing = "unsloth", 
    random_state = 3407,
)

## 4. Load the SFT Traces Dataset
We load the dataset you compiled which contains `pi-mono`, `vojtavlas2`, and your personalized traces!

In [ ]:
from datasets import load_dataset

dataset = load_dataset("focustiki/sft-coding-agent-traces")
train_data = dataset["train"]
eval_data = dataset["test"]

print(f"Loaded {len(train_data)} training examples and {len(eval_data)} eval examples.")

## 5. Apply the Chat Template
We format the structured JSON array into the standard `assistant`, `user`, and `tool` chat template recognized by Gemma.

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma",
)

def formatting_prompts_func(examples):
    texts = []
    for messages in examples["messages"]:
        # The tokenizer automatically applies the chat format and <bos>/<eos> tokens
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return { "text" : texts, }

train_dataset = train_data.map(formatting_prompts_func, batched = True)
eval_dataset = eval_data.map(formatting_prompts_func, batched = True)

## 6. Train the Model! 🚀

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # max_steps = 60, # Uncomment to do a quick 60-step test run!
        num_train_epochs = 1, # Set to 3 for a full training run
        learning_rate = 2e-5,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

## 7. Save and Publish to Hugging Face
Once training is done, we push the adapter weights back to your HF account so you can use them locally!

In [ ]:
model.push_to_hub("focustiki/gemma-4-4b-coder-sft", token = hf_token)
tokenizer.push_to_hub("focustiki/gemma-4-4b-coder-sft", token = hf_token)

print("✅ Successfully pushed to Hugging Face!")